# Hindi ModernBERT — pipeline map

Run cells **top to bottom**. Each stage has:
1. A markdown cell explaining what happens in production
**Kernel cwd:** repo root or `notebook/` (auto-detects repo).

**Kernel cwd:** repo root (`indic-modernBERT/`).

```mermaid
flowchart LR
  A[pretokenization] --> B[train-bpe]
  B --> C[eval-bpe]
  C --> D[configs]
  D --> E[tokenize + MLM]
  E --> F[GPU batch]
  F --> G[model forward]
  G --> H[backward]
  H --> I[train-mlm-probe]
  I --> J[train-pretrain 22L]
```


In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))  # notebook/pipeline_steps.py

from pipeline_steps import (
    PipelineContext,
    StageSkip,
    detect_repo,
    ensure_src_on_path,
    print_stage_result,
    validate_attention,
    validate_config,
    validate_data,
    validate_environment,
    validate_full_pretrain,
    validate_gpu,
    validate_model_smoke,
    validate_probe,
    validate_tokenizer,
    validate_trace,
)

REPO = detect_repo()
ensure_src_on_path(REPO)  # adds src to path + chdir to repo root
ctx = PipelineContext.detect()


def run_stage(step: int, title: str, fn):
    """Run one pipeline stage; print ✓ with details or ⊘ if optional dep missing."""
    try:
        result = fn(ctx)
        print_stage_result(step, title, result)
    except StageSkip as exc:
        print(f"⊘ Step {step} — {title} SKIPPED: {exc}")


print("REPO:", ctx.repo)
print("device:", ctx.device)
print("tokenizer artifact:", ctx.has_tokenizer())
print("train parquet:", ctx.has_train_parquet())


/home/krrish/Desktop/Programming/indic-modernBERT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPO: /home/krrish/Desktop/Programming/indic-modernBERT
device: cuda
tokenizer artifact: True
train parquet: True


## Step 0 — Environment

Production training needs a CUDA GPU with **bf16** and **flash-attn** for sliding-window attention.
Validates `resolve_device`, GPU summary, and `IMPL_USE_FLASH2`.
Related CLI: none (implicit in all GPU targets).


In [2]:
run_stage(0, "Environment", validate_environment)


✓ Step 0 — Environment
    device: cuda
    summary: device=cuda:None name=NVIDIA GeForce RTX 4090 cc=(8, 9) mem=25.4GB bf16=True
    flash_attn: True


## Step 1 — Model + job configs

Production architecture: **22-layer `modernbert_base`** (unpadded, FA2, sliding window 128, global every 3).
Hydra configs `hindi_mlm_phase1` + `hindi_mlm_smoke_50ba` load Pydantic `PretrainConfig`
(`max_seq_len=1024`, dataloader worker / prefetch settings).
Files: `configs/model/modernbert_base.yaml`, `configs/pretrain/hindi_mlm_*.yaml`.


In [3]:
run_stage(1, "Model + job configs", validate_config)


✓ Step 1 — Model + job configs
    arch: 22L hidden=768 window=128 global_every=3
    vocab: 50368
    phase1: lr=0.0001 global_batch=512 max_seq=1024
    production: hindi_mlm_smoke_50ba
    workers: train=2 prefetch=4 eval=3 packing_prefetch=5
    packing: enabled prefetch=5 micro=8


## Step 2 — Data layer

Sangrah parquet shards → text rows. Pretokenization applies Hindi script normalization.
Production: `make pretokenization` before BPE training.
Validates `preprocess_for_tokenizer`, `ParquetMLMDataset`, `load_eval_texts`, and when the
tokenizer artifact exists: raw train, packed train, and eval `DataLoader`s with production
`num_workers` / `prefetch_factor` from `hindi_mlm_smoke_50ba`.

**Batch fields:** raw train collator returns `input_ids` only (MLM happens in the packer).
Eval and packed loaders return `input_ids`, `attention_mask`, and `labels` (`-100` = no loss).
The stage prints keys, token/pad/mask counts, decoded text, and MLM mask previews.


In [4]:
run_stage(2, "Data layer", validate_data)


2026-06-14 22:42:22.271 | INFO     | pretrain.step_log:step_log:19 - data | tokenize start | #1 | n_texts=512 | max_seq_len=1024
2026-06-14 22:42:22.271 | INFO     | pretrain.step_log:step_log:19 - data | tokenize start | #1 | n_texts=512 | max_seq_len=1024
2026-06-14 22:42:22.635 | INFO     | pretrain.step_log:step_log:19 - data | tokenize done | #1 | nonempty=512 | len min=54 max=1024 avg=419 | 0.4s
2026-06-14 22:42:22.642 | INFO     | pretrain.step_log:step_log:19 - data | tokenize done | #1 | nonempty=512 | len min=59 max=1024 avg=433 | 0.4s
2026-06-14 22:42:22.681 | INFO     | pretrain.step_log:step_log:19 - data | tokenize start | #2 | n_texts=512 | max_seq_len=1024
2026-06-14 22:42:22.687 | INFO     | pretrain.step_log:step_log:19 - data | tokenize start | #2 | n_texts=512 | max_seq_len=1024
2026-06-14 22:42:23.054 | INFO     | pretrain.step_log:step_log:19 - data | tokenize done | #2 | nonempty=512 | len min=50 max=1024 avg=420 | 0.3s
2026-06-14 22:42:23.076 | INFO     | pretra

    ↳ raw [attention_mask, input_ids]: 'अब  ग्राहक  अपने  स्मार्टफोन  में  The  F u el  Del iv ery  मोबाइल  ऐप  को  डाउनलोड  करके  फ्यूल  ऑर्डर  और  डिलीवरी  की  मॉनिटरिंग  कर  सकत'
    ↳ eval [attention_mask, input_ids, labels]: len=1024 real=426 pad=598 masked=58
    ↳ eval decoded: 'उत्तर  प्रदेश  के  ( it rak oot )  गुरुवार  को  प्रेमी  जोड़े  ने  आत्म हा त्य  कर  ली  ( C ou ple  Su icide Ẓ  के'
    ↳ eval MLM masks: '[MASK]'->' चित्रकूट', '[MASK]'->'Ch', '[MASK]'->' में', '[MASK]'->' एक'
    ↳ packed [attention_mask, cu_seqlens, input_ids, labels, max_seqlen]: len=8192 real=8174 masked=2503
    ↳ packed MLM masks: '[MASK]'->' स्मार्टफोन', '[MASK]'->'u', '[MASK]'->' को', '[MASK]'->' करके'
✓ Step 2 — Data layer
    preprocess: '  भारत   '
    list_dataset: len=2
    describe: 274 parquet shard(s) under /home/krrish/Desktop/Programming/indic-modernBERT/data/sangrah_dataset
    parquet: shard=data-0.parquet sample_len=2476 eval_rows=4
    pretrain_config: hindi_mlm_smoke_50ba
    trai

2026-06-14 22:42:34.597 | INFO     | pretrain.step_log:step_log:19 - data | tokenize done | #9 | nonempty=512 | len min=40 max=1024 avg=426 | 0.4s
2026-06-14 22:42:34.635 | INFO     | pretrain.step_log:step_log:19 - data | tokenize done | #9 | nonempty=512 | len min=39 max=1024 avg=444 | 0.5s


## Step 3 — Tokenizer + MLM masking

BPE tokenizer artifact (`vocab=50368`) is loaded and batches are masked for MLM using
production probabilities from `hindi_mlm_smoke_50ba` (`mlm_probability` / `eval_mlm_probability`)
at `max_seq_len=1024`.
Production: `make train-bpe` then `artifacts/tokenizer/bpe_vs50368/`.
Prints a Hindi sample, its tokenization, and masked token pairs from the collator.


In [5]:
run_stage(3, "Tokenizer + MLM masking", validate_tokenizer)


    ↳ sample: 'भारत एक विशाल देश है और हिंदी इसकी प्रमुख भाषाओं में से एक है।'
    ↳ tokenized: 'भारत  एक  विशाल  देश  है  और  हिंदी  इसकी  प्रमुख  भाषाओं  में  से  एक  है ।'
    ↳ mlm masks (train 0.3): ' चयाप'->' एक', '[MASK]'->' देश', '[MASK]'->' और', '[MASK]'->' एक'
✓ Step 3 — Tokenizer + MLM masking
    tokenizer_dir: /home/krrish/Desktop/Programming/indic-modernBERT/artifacts/tokenizer/bpe_vs50368
    vocab: 50368
    max_seq_len: 1024
    tokenize_batch_shape: (1, 1024)
    mlm_masks: train=4 eval=3 (0.3 / 0.15 prob)
    sample_text: भारत एक विशाल देश है और हिंदी इसकी प्रमुख भाषाओं में से एक है।
    tokenized_sample: भारत  एक  विशाल  देश  है  और  हिंदी  इसकी  प्रमुख  भाषाओं  में  से  एक  है ।
    masked_preview: ' चयाप'->' एक', '[MASK]'->' देश', '[MASK]'->' और', '[MASK]'->' एक'


2026-06-14 22:42:34.512 | INFO     | pretrain.step_log:step_log:19 - data | packer yield | #4 | packed_rows=64 tokens=523383 mlm_masked=157383


## Step 4 — GPU batch helpers

Unpadded RoPE models need `position_ids`. Training uses bf16 autocast on CUDA.
Functions: `add_position_ids`, `move_batch_to_device`, `training_autocast` in `pretrain/gpu_batch.py`.


In [6]:
run_stage(4, "GPU batch helpers", validate_gpu)


✓ Step 4 — GPU batch helpers
    position_ids_row0: [0, 1, 2, 0, 0]
    batch_device: cuda:0
    autocast_dtype: torch.bfloat16


## Step 5 — Model smoke (modernbert_base)

Loads the **production** 22-layer `modernbert_base.yaml` model (same arch as phase-1 pretrain):
unpadded, RoPE, FA2, sliding window 128, global attention every 3 layers, vocab 50368.
Runs a single Hindi MLM forward pass at production `max_seq_len=1024`.


In [7]:
run_stage(5, "Model smoke (22L base)", validate_model_smoke)


[transformers] FlexBertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
2026-06-14 22:42:35.708 | INFO     | pretrain.step_log:step_log:19 - data | packer yield | #5 | packed_rows=64 tokens=523266 mlm_masked=157400
2026-0

✓ Step 5 — Model smoke (22L base)
    arch: 22L hidden=768 padding=unpadded
    vocab: 50368
    max_seq_len: 1024
    attention: fa2=True window=128 global_every=3
    forward: loss=11.2914 logits=(4, 50368)


## Step 6 — Attention verification

Checks **22L** per-layer pattern (global vs sliding: L0 global, L1–L2 sliding, L3 global, …),
MLM logits shape, and backward gradients on Hindi text at `max_seq_len=1024`.
Forward/backward use production `device_train_microbatch_size` from `hindi_mlm_smoke_50ba`.
CLI equivalent: `make pipeline-trace`.


In [8]:
run_stage(6, "Attention verification", validate_attention)


[transformers] FlexBertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


✓ Step 6 — Attention verification
    layers: 22L global=8 sliding=14
    pattern_head: L0:global, L1:sliding, L2:sliding, L3:global
    layer0: FlexBertUnpadRopeAttention window=(-1, -1)
    microbatch: n=2 seq=1024
    mlm_backward: loss=10.8202 grad_tensors=137


## Step 7 — Production training step

Builds **22L `modernbert_base`**; runs forward, `evaluate_mlm`, and `loss.backward()` using
production `build_eval_dataloader` (global eval batch 64, microbatch 16) and
`build_parquet_train_dataloader` from `hindi_mlm_smoke_50ba`.
Production: `make train-smoke-50ba`. LR sweep: `make lr-sweep`.


In [9]:
run_stage(7, "Production training step", validate_probe)


[transformers] FlexBertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


2026-06-14 22:42:47.032 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm start | #1 | n_texts=64 | max_seq_len=1024 prob=0.15
2026-06-14 22:42:47.034 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm start | #1 | n_texts=64 | max_seq_len=1024 prob=0.15
2026-06-14 22:42:47.039 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm start | #1 | n_texts=64 | max_seq_len=1024 prob=0.15
2026-06-14 22:42:47.127 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm done | #1 | shape=(64, 1024) masked=4455 | 0.1s
2026-06-14 22:42:47.127 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm done | #1 | shape=(64, 1024) masked=3565 | 0.1s
2026-06-14 22:42:47.131 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm done | #1 | shape=(64, 1024) masked=4369 | 0.1s
2026-06-14 22:42:47.134 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm start | #2 | n_texts=64 | max_seq_len=1

✓ Step 7 — Production training step
    arch: 22L vocab=50368
    device: cuda:0
    eval_loader: workers=3 batch=(64, 1024) micro=8
    eval_keys: attention_mask, input_ids, labels
    eval_tokens: real=426 masked=59
    eval_sample: उत्तर  प्रदेश  के Ch it rak oot )  गुरुवार  को  एक  प्रेमी  जोड़े  ने  आत्म हा त्य  कर  ली  ( asi ou ple icide ).  पुलिस  के
    eval_masked_preview: '[MASK]'->' चित्रकूट', '[MASK]'->' (', '[MASK]'->' में', 'asi'->'C'
    forward: loss=10.8940 logits=(374, 50368) acc=0.0000
    evaluate: loss=10.8846 acc=0.0000 tokens=1077
    packed_batch: shape=(64, 8192)
    backward: loss=10.8944 grad_tensors=137


## Step 8 — End-to-end trace

Single-batch path: GPU → tokenize+mask (`max_seq_len=1024`) → **22L model config** → forward → backward.
Uses production microbatch sizing. CLI equivalent: `make pipeline-trace`.


In [10]:
run_stage(8, "End-to-end trace", validate_trace)


[transformers] FlexBertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


✓ Step 8 — End-to-end trace
    gpu: device=cuda:None name=NVIDIA GeForce RTX 4090 cc=(8, 9) mem=25.4GB bf16=True | tensor_device=cuda:0 bf16_supported=True; flash_attn=True
    tokenize+mlm_mask: batch shape (2, 1024) | mlm_masked=5 pad_tokens=2017 | input_ids.device=cuda:0
    model_config: layers=22 padding=unpadded attn=rope use_fa2=True sliding_window=(-1, -1) class=FlexBertUnpadRopeAttention
    forward+logits: loss=10.9916 logits=(5, 50368) loss_device=cuda:0
    backward: loss=10.9974 params_with_grad=137 grad_max=1.861e+01
    arch: modernbert_base
    max_seq_len: 1024


## Step 9 — Full pretrain (manual e2e loop)

Runs **3 packed train steps** end-to-end: sample from `build_parquet_train_dataloader` → `split_packed_batch` → forward → backward → `optimizer.step()` → padded eval forward + `evaluate_mlm`.

Same production stack as `hindi_mlm_smoke_50ba` (22L, FA2, torch.compile, `decoupled_stableadamw`). Requires CUDA + `uv sync --extra pretrain`.

**Notebook output:** this cell prints live `Step 9 | ...` progress lines as it runs (scroll the cell output below the code). The first forward may take **1–3 min** during `torch.compile` — wait for `first forward+backward` to finish.

Composer production runs (terminal, full logs in `logs/smoke_50ba/train.log`):

- **Smoke (50 batches):** `make train-smoke-50ba`
- **Phase-1 pretrain:** `run_pretrain.py --config-name hindi_mlm_phase1`
- **LR sweep:** `make lr-sweep`


In [ ]:
run_stage(9, "Full pretrain (manual e2e loop)", validate_full_pretrain)


Step 9 | device=cuda | global_batch=8 micro=8 | 3 train steps + eval
Step 9 | loading 22L model (bert-base init + tokenizer vocab)...


[transformers] FlexBertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Step 9 | validating production kernels (FA2, compile, loss fn)...


2026-06-14 22:43:07.445 | INFO     | pretrain.train:_validate_production_kernels:66 - Production kernels OK | FA2 installed | compile_model=true | attention layers FA3=0 FA2=22 | loss=fa_cross_entropy


Step 9 | building packed train + eval dataloaders (workers=0)...
Step 9 | train step 1/3: fetching packed batch...


...

Step 9 | train step 3/3: packed=(1, 8192) seqs=27 microbatches=1
Step 9 | train step 3/3: micro 1/1 loss=10.3398
Step 9 | train step 3/3: optimizer.step() mean_loss=10.3398
Step 9 | eval forward (micro=8)...


2026-06-14 22:43:11.024 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm start | #1 | n_texts=8 | max_seq_len=1024 prob=0.15
2026-06-14 22:43:11.047 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm done | #1 | shape=(8, 1024) masked=405 | 0.0s


Step 9 | eval forward: loss=10.2010 masked=405 acc=0.0321
Step 9 | evaluate_mlm (2 batches)...


2026-06-14 22:43:11.064 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm start | #2 | n_texts=8 | max_seq_len=1024 prob=0.15
2026-06-14 22:43:11.080 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm done | #2 | shape=(8, 1024) masked=424 | 0.0s
2026-06-14 22:43:11.095 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm start | #3 | n_texts=8 | max_seq_len=1024 prob=0.15
2026-06-14 22:43:11.109 | INFO     | pretrain.step_log:step_log:19 - data | eval tokenize+mlm done | #3 | shape=(8, 1024) masked=562 | 0.0s


Step 9 | eval loop: loss=10.1470 acc=0.0458 tokens=986
✓ Step 9 — Full pretrain (manual e2e loop)
    path: packed train → 3× (forward+backward+optim.step) → eval
    config: hindi_mlm_smoke_50ba, 1 shard, workers=0, no packer warmup
    arch: 22L loss=fa_cross_entropy
    batch: global=8 micro=8 grad_accum=1
    steps: step1: micros=1 seqs=23 loss=10.8696 grads=137 | step2: micros=1 seqs=29 loss=10.5656 grads=137 | step3: micros=1 seqs=27 loss=10.3398 grads=137
    packed_batch: (1, 8192)
    last_micro: masked=2456 logits=(2456, 50368)
    eval_forward: loss=10.2010 masked=405 logits=(405, 50368) acc=0.0321
    eval_loop: loss=10.1470 acc=0.0458 batches=2 tokens=986
    production_entry: scripts/run_pretrain.py
    production_smoke: make train-smoke-50ba
    production_full: run_pretrain.py --config-name hindi_mlm_phase1


## Bonus — MLM mask visualization

Inspect which token positions receive MLM loss (`labels != -100`) after the collator.


In [ ]:
from transformers import DataCollatorForLanguageModeling, PreTrainedTokenizerFast
from pretrain.gpu_batch import move_batch_to_device
from pretrain.parquet_mlm import load_eval_texts
from utils.paths import resolve_hf_tokenizer_dir

if not ctx.has_tokenizer():
    print("⊘ SKIPPED — complete Step 3 first")
else:
    tok = PreTrainedTokenizerFast.from_pretrained(str(resolve_hf_tokenizer_dir(ctx.tokenizer_path)))
    texts = (
        load_eval_texts(
            ctx.eval_data if ctx.has_eval_parquet() else ctx.train_data,
            "text",
            max_rows=2,
        )
        if (ctx.has_eval_parquet() or ctx.has_train_parquet())
        else ["परीक्षण वाक्य"]
    )
    enc = tok(texts[:1], padding="max_length", truncation=True, max_length=64)
    ex = [{k: enc[k][0] for k in enc}]
    batch = move_batch_to_device(
        DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=0.3)(ex),
        ctx.device,
    )
    masked = int((batch["labels"] != -100).sum())
    assert masked > 0, "expected at least one MLM mask"
    print(f"✓ {masked} masked positions in batch")
    for i in range(batch["input_ids"].shape[1]):
        lid = int(batch["labels"][0, i])
        if lid == -100:
            continue
        tid = int(batch["input_ids"][0, i])
        print(f"  pos {i:2d} | id={tid:5d} | token={tok.decode([tid])!r} | label={lid}")


✓ 25 masked positions in batch
  pos  5 | id=    4 | token='[MASK]' | label=2615
  pos 11 | id=    4 | token='[MASK]' | label=2513
  pos 16 | id=    4 | token='[MASK]' | label=10461
  pos 18 | id=    4 | token='[MASK]' | label=4358
  pos 21 | id=    4 | token='[MASK]' | label=2530
  pos 22 | id=    4 | token='[MASK]' | label=3247
  pos 26 | id=    4 | token='[MASK]' | label=16609
  pos 27 | id=14454 | token='emb' | label=15330
  pos 28 | id=    4 | token='[MASK]' | label=44305
  pos 29 | id=    4 | token='[MASK]' | label=14545
  pos 30 | id=    4 | token='[MASK]' | label=2869
  pos 34 | id=    4 | token='[MASK]' | label=8183
  pos 36 | id=    4 | token='[MASK]' | label=16521
  pos 41 | id=    4 | token='[MASK]' | label=7889
  pos 43 | id=    4 | token='[MASK]' | label=2505
  pos 44 | id=    4 | token='[MASK]' | label=18822
  pos 45 | id=    4 | token='[MASK]' | label=2615
  pos 46 | id=    4 | token='[MASK]' | label=11958
  pos 50 | id=    4 | token='[MASK]' | label=16
  pos 51 | id=  

: 